# `helpers/calendars.py` — Playground

Manual verification notebook for all calendar helpers.

| Function | Status | Notes |
|---|---|---|
| `get_upcoming_macro_events(hours_ahead)` | ✅ built | |
| `get_hours_to_next_macro_event()` | ✅ built | |
| `get_ticker_earnings_window(ticker)` | ✅ built | |

In [ ]:
import sys
import pathlib

helpers_dir = pathlib.Path('backend/02_intelligence/helpers').resolve()
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from calendars import (
    get_upcoming_macro_events,
    get_hours_to_next_macro_event,
    get_ticker_earnings_window,
)

## `get_upcoming_macro_events(hours_ahead=24)`

**Expected output:** list of dicts with `event`, `country`, `time`, `impact`  
**Gate 1 block rule:** any event returned → block (FOMC/CPI/NFP within 24h)

In [ ]:
events_24h = get_upcoming_macro_events(hours_ahead=24)
print(f'Events in next 24h: {len(events_24h) if events_24h is not None else "FETCH FAILED"}')
for e in (events_24h or []):
    print(f"  {e['time']} UTC  {e['event']}  [{e['impact']}]")

In [ ]:
# Look 7 days ahead to confirm filtering is working
events_7d = get_upcoming_macro_events(hours_ahead=168)
assert events_7d is not None, "Fetch failed — check FINNHUB_API_KEY"
print(f'Events in next 7 days: {len(events_7d)}')
for e in events_7d:
    print(f"  {e['time']} UTC  {e['event']}  [{e['impact']}]")
print('\n✅ Fetch succeeded')

---
## `get_hours_to_next_macro_event()`

**Expected output:** float (hours until nearest event) or None  
**Used by:** `get_market_context()` in `market.py` → Gate 4 prompt

In [ ]:
hours = get_hours_to_next_macro_event()
if hours is not None:
    print(f'Next macro event in: {hours} hours ({hours/24:.1f} days)')
else:
    print('No upcoming macro event found within 7 days (or fetch failed)')
print('\n✅ Function returned without error')

---
## `get_ticker_earnings_window(ticker, days_ahead)`

**Returns:** `{reports_today, reports_tomorrow, report_date, hour}` or `None` on fetch failure.

`hour` values:
- `'amc'` — after market close (after 4 PM ET) — most common
- `'bmo'` — before market open (before 9:30 AM ET)
- `'dmh'` — during market hours
- `None` — unknown timing

**Gate 1 block rule:** block if `reports_today` or `reports_tomorrow` is True.

In [ ]:
# Happy path — 90-day window guarantees a result for AAPL
window = get_ticker_earnings_window('AAPL', days_ahead=90)
print(f'reports_today:    {window["reports_today"]}')
print(f'reports_tomorrow: {window["reports_tomorrow"]}')
print(f'report_date:      {window["report_date"]}')
print(f'hour:             {window["hour"]}')

In [ ]:
# Parameter variation — compare two tickers over 1-day window
for ticker in ['AAPL', 'MSFT', 'NVDA']:
    w = get_ticker_earnings_window(ticker, days_ahead=1)
    flag = '🚨 BLOCK' if (w and (w['reports_today'] or w['reports_tomorrow'])) else '✅ clear'
    print(f'{ticker:6s}  today={w["reports_today"]}  tomorrow={w["reports_tomorrow"]}  {flag}')

In [ ]:
# Failure path — unknown ticker returns dict with False/None (not None)
bad = get_ticker_earnings_window('ZZZZINVALID', days_ahead=1)
assert bad is not None, 'Expected dict, got None'
assert bad['reports_today'] is False
assert bad['report_date'] is None
print(f'Unknown ticker result: {bad}')
print('✅ unknown ticker correctly returned empty dict')

In [ ]:
# Free play — try any ticker or window here
